In [3]:
# Setup: run this cell first in every session
import openml, hashlib, os
import numpy as np
import pandas as pd

df = pd.read_parquet("data/raw_public_procurement.parquet")
print(df.shape)

(565163, 75)


In [1]:
# Import required libraries
import openml, hashlib
import numpy as np
import pandas as pd
print("ready")

ready


In [3]:
# Download the dataset from OpenML (ID 42163)
dataset = openml.datasets.get_dataset(42163)
df, _, _, _ = dataset.get_data()

# Basic checks
print("Dataset name:", dataset.name)
print("Version:", dataset.version)
print("Shape (rows, columns):", df.shape)

Dataset name: public_procurement
Version: 6
Shape (rows, columns): (565163, 75)


In [4]:
import os

# Save the raw data to a local file
os.makedirs("data", exist_ok=True)
raw_path = "data/raw_public_procurement.parquet"
df.to_parquet(raw_path, index=False)

# Compute the MD5 checksum of the saved file
with open(raw_path, "rb") as f:
    md5 = hashlib.md5(f.read()).hexdigest()
print("MD5:", md5)

MD5: 6a72ecce154c09b1956bb87de94e9443


In [1]:
# Reload raw data from the saved file (no need to download again)
import openml, hashlib, os
import numpy as np
import pandas as pd

df = pd.read_parquet("data/raw_public_procurement.parquet")
print(df.shape)

(565163, 75)


In [2]:
# Build an overview table: one row per column
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "pct_missing": (df.isna().mean() * 100).round(1),
    "pct_minus1": ((df == -1).mean() * 100).round(1),
    "example": df.apply(lambda c: str(c.dropna().iloc[0])[:40] if c.notna().any() else None),
})
summary.to_csv("data/column_overview.csv")
print(summary.to_string())

                                dtype  n_unique  pct_missing  pct_minus1                                   example
id_notice_can                   int64    172594          0.0         0.0                                     20155
ted_notice_url                    str    172594          0.0         0.0  ted.europa.eu/udl?uri=TED:NOTICE:5-2015:
year                            int64         1          0.0         0.0                                      2015
id_type                         uint8         3          0.0         0.0                                         3
dt_dispatch                       str       376          0.0         0.0                                 23-DEC-14
xsd_version                       str         2          0.0         0.0                                   R208.S2
cancelled                       uint8         1          0.0         0.0                                         0
corrections                     uint8         1          0.0         0.0        

In [9]:
# Column decisions: (decision, reason) -> list of columns
TARGET = "award_value_euro"
GROUP_ID = "id_notice_can"

decisions = {
    ("keep", "Pre-tender feature"): [
        "id_type", "cae_gpa_annex", "iso_country_code", "cae_type", "main_activity",
        "b_on_behalf", "type_of_contract", "tal_location_nuts", "b_fra_agreement",
        "b_fra_contract", "cpv", "b_gpa", "gpa_coverage", "lots_number", "b_eu_funds",
        "top_type", "out_of_directives", "crit_code", "b_electronic_auction"],
    ("target", "Target variable; -1 recoded as missing"): [TARGET],
    ("drop", "Identifier (notice ID kept separately for group split)"): [
        GROUP_ID, "ted_notice_url", "id_award", "id_lot_awarded", "contract_number", "cae_nationalid"],
    ("drop", "Post-award information (data leakage)"): [
        "dt_dispatch", "dt_award", "value_euro", "value_euro_fin_1", "value_euro_fin_2",
        "award_value_euro_fin_1", "number_awards", "number_offers", "number_offers_electr",
        "number_tenders_sme", "number_tenders_other_eu", "number_tenders_non_eu",
        "win_name", "win_nationalid", "win_address", "win_town", "win_postal_code",
        "win_country_code", "b_contractor_sme", "b_awarded_to_a_group", "b_subcontracted",
        "info_on_non_award"],
    ("drop", "Free text or location text with very high cardinality"): [
        "cae_name", "cae_address", "cae_town", "cae_postal_code", "title",
        "crit_criteria", "crit_weights", "additional_cpvs"],
    ("drop", "Constant or mostly missing"): [
        "year", "cancelled", "corrections", "b_multiple_cae", "b_multiple_country",
        "iso_country_code_all", "eu_inst_code", "b_involves_joint_procurement",
        "b_awarded_by_central_body", "id_lot", "b_dyn_purch_syst", "b_accelerated",
        "fra_estimated", "crit_price_weight", "info_unpublished"],
    ("drop", "Technical metadata"): ["xsd_version"],
    ("drop", "Redundant with iso_country_code"): ["iso_country_code_gpa"],
    ("drop", "Derived from CPV code (2-digit division, 3-digit for division 79); redundant"): ["main_cpv_code_gpa"],
    ("drop", "Pre-tender estimate; excluded from main model (log corr. 0.928 with target)"): ["award_est_value_euro"],
}

In [10]:
# Flatten into a table: one row per column
rows = [(col, d, r) for (d, r), cols in decisions.items() for col in cols]
table = pd.DataFrame(rows, columns=["column", "decision", "reason"])

# Sanity checks: every column covered exactly once
print("Not covered:", set(df.columns) - set(table["column"]))
print("Duplicates:", table.loc[table["column"].duplicated(), "column"].tolist())
print(table["decision"].value_counts())

# Add the overview statistics and save
overview = pd.read_csv("data/column_overview.csv", index_col=0)
table = table.merge(overview, left_on="column", right_index=True)
os.makedirs("reports", exist_ok=True)
table.to_csv("reports/column_decisions.csv", index=False)

Not covered: set()
Duplicates: []
decision
drop      55
keep      19
target     1
Name: count, dtype: int64


In [6]:
# Check 1: is main_cpv_code_gpa the same as the CPV division?
cpv_div = df["cpv"].astype(str).str.zfill(8).str[:2].astype(int)
same = (cpv_div == df["main_cpv_code_gpa"])
print("CPV division == main_cpv_code_gpa:", round(same.mean() * 100, 1), "%")
print(df.loc[~same, ["cpv", "main_cpv_code_gpa"]].head(10))

# Check 2: how close is the estimated value to the actual award value?
target = df["award_value_euro"].replace(-1, np.nan)
est = df["award_est_value_euro"]
both = est.notna() & target.notna() & (est > 0) & (target > 0)
print("\nRows with both values:", both.sum())
print("Exactly equal:", round((est[both] == target[both]).mean() * 100, 1), "%")
print("Log correlation:", round(np.corrcoef(np.log(est[both]), np.log(target[both]))[0, 1], 3))

CPV division == main_cpv_code_gpa: 89.6 %
         cpv  main_cpv_code_gpa
2   79620000                796
3   79210000                792
4   79210000                792
8   79410000                794
9   79410000                794
10  79410000                794
37  79100000                791
38  79100000                791
39  79100000                791
40  79100000                791

Rows with both values: 199624
Exactly equal: 14.5 %
Log correlation: 0.928


In [11]:
# Step 1: recode -1 as missing in the target, then drop rows without a target
clean = df.copy()
clean[TARGET] = clean[TARGET].replace(-1, np.nan)

rows_before = len(clean)
clean = clean.dropna(subset=[TARGET])

print("Rows before:", rows_before)
print("Rows removed:", rows_before - len(clean))
print("Rows remaining:", len(clean))
print("Zero or negative target values:", (clean[TARGET] <= 0).sum())

Rows before: 565163
Rows removed: 199954
Rows remaining: 365209
Zero or negative target values: 6


In [12]:
# Step 2: inspect and remove invalid (zero or negative) target values
invalid = clean[clean[TARGET] <= 0]
print(invalid[[TARGET, "iso_country_code", "cpv", "type_of_contract", "top_type"]])

clean = clean[clean[TARGET] > 0]
print("\nRows remaining:", len(clean))

        award_value_euro iso_country_code       cpv type_of_contract top_type
125528               0.0               RO  33600000                U      OPE
125529               0.0               RO  33600000                U      OPE
410107               0.0               RO  33651100                U      OPE
456177               0.0               RO  33141641                U      OPE
456179               0.0               RO  33141641                U      OPE
519948               0.0               PL  66110000                S      OPE

Rows remaining: 365203


In [13]:
# Step 3: save group IDs, add derived columns, drop unused columns
clean = clean.reset_index(drop=True)
clean[[GROUP_ID]].to_parquet("data/group_ids.parquet", index=False)

clean["cpv_div"] = clean["cpv"].astype(str).str.zfill(8).str[:2]
clean["nuts2"] = clean["tal_location_nuts"].str[:4]
clean["b_fra_contract"] = clean["b_fra_contract"].fillna("N")

drop_cols = table.loc[table["decision"] == "drop", "column"].tolist()
clean = clean.drop(columns=drop_cols)

print("Shape:", clean.shape)
print(clean.columns.tolist())

Shape: (365203, 22)
['id_type', 'cae_gpa_annex', 'iso_country_code', 'cae_type', 'main_activity', 'b_on_behalf', 'type_of_contract', 'tal_location_nuts', 'b_fra_agreement', 'b_fra_contract', 'cpv', 'b_gpa', 'gpa_coverage', 'lots_number', 'b_eu_funds', 'top_type', 'out_of_directives', 'crit_code', 'b_electronic_auction', 'award_value_euro', 'cpv_div', 'nuts2']


In [14]:
# Step 4: final checks, then save the clean dataset
ids = pd.read_parquet("data/group_ids.parquet")
print("Same length as group IDs:", len(ids) == len(clean))
print("Unique notices:", ids[GROUP_ID].nunique())
print("Missing target:", clean[TARGET].isna().sum())
print("Target min / max:", clean[TARGET].min(), "/", clean[TARGET].max())
print("CPV divisions (first 5):", sorted(clean["cpv_div"].unique())[:5])

clean_path = "data/clean_public_procurement.parquet"
clean.to_parquet(clean_path, index=False)
with open(clean_path, "rb") as f:
    print("Clean MD5:", hashlib.md5(f.read()).hexdigest())

Same length as group IDs: True
Unique notices: 111479
Missing target: 0
Target min / max: 0.01 / 13121471674.2
CPV divisions (first 5): ['03', '09', '14', '15', '16']
Clean MD5: 0eab8eb71af6503ede35b2e461a1041d
